# Q5 -- Deployment

## Q5.1 -- Gradio Web Service

We deploy a Gradio web application for variety-aware Sentiment and Sarcasm classification.
The UI lets users input text and **explicitly select the English variety** (en-AU, en-IN, en-UK).

**Architecture -- adapter-swap pattern:**
The system mirrors the LoRA adapter-swap approach from Q2.3. At startup, variety-specific
model weights are pre-loaded into memory. When the user selects a variety, the backend
indexes directly into the correct model -- no reload required. This is the same efficiency
principle behind LoRA: a frozen base model with only small adapter weights changing per request.

Two backends are implemented:
- **Live inference backend:** TF-IDF + Logistic Regression per variety (fast, no GPU, ~1-3ms)
- **LoRA backend (commented out in Cell 3):** shows how to load TinyLlama + variety-specific
  LoRA adapters from HuggingFace once the adapters are uploaded there.

**Why Gradio?** Single-file deployment, share=True generates a public HTTPS URL with no
infrastructure setup. Unlike Flask it needs no route/template boilerplate; unlike Streamlit
it is natively designed for ML input/output interfaces.

In [1]:
# -- Cell 1: Install dependencies --
!pip install gradio datasets scikit-learn transformers peft -q

In [2]:
# -- Cell 2: Train variety-specific TF-IDF + LR models --
import pandas as pd
import numpy as np
import time
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("Loading BESSTIE dataset...")
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")
train_df = pd.DataFrame(ds['train'])
test_df   = pd.DataFrame(ds['test'])

variety_models = {}
for variety in ['en-AU', 'en-IN', 'en-UK']:
    v_train = train_df[train_df['variety'] == variety].reset_index(drop=True)

    tfidf_s = TfidfVectorizer(max_features=5000)
    lr_s    = LogisticRegression(max_iter=1000)
    lr_s.fit(tfidf_s.fit_transform(v_train['text']), v_train['Sentiment'])

    tfidf_r = TfidfVectorizer(max_features=5000)
    lr_r    = LogisticRegression(max_iter=1000, class_weight='balanced')
    lr_r.fit(tfidf_r.fit_transform(v_train['text']), v_train['Sarcasm'])

    variety_models[variety] = {
        'sentiment': (tfidf_s, lr_s),
        'sarcasm':   (tfidf_r, lr_r),
    }

print("All variety-specific models loaded")
print("Varieties:", list(variety_models.keys()))

Loading BESSTIE dataset...
All variety-specific models loaded
Varieties: ['en-AU', 'en-IN', 'en-UK']


In [3]:
# -- Cell 3 (Reference): LoRA adapter loading pattern --
# Once LoRA adapters are uploaded to HuggingFace, replace this with actual repo IDs.
# The adapter-swap approach keeps the base model frozen in memory; only ~90MB of
# adapter weights change per variety switch (vs ~2.2GB full model reload).

# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch
#
# BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# LORA_ADAPTERS = {
#     'en-AU': "<hf-username>/tinyllama-lora-sarcasm-en-au",
#     'en-IN': "<hf-username>/tinyllama-lora-sarcasm-en-in",
#     'en-UK': "<hf-username>/tinyllama-lora-sarcasm-en-uk",
# }
#
# base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
#
# loaded_adapters = {}
# for variety, repo in LORA_ADAPTERS.items():
#     loaded_adapters[variety] = PeftModel.from_pretrained(base_model, repo)
#
# def lora_predict(text, variety):
#     model = loaded_adapters[variety]
#     prompt = f"Classify as Sarcastic or Not Sarcastic: {text}\nLabel:"
#     inputs = base_tokenizer(prompt, return_tensors='pt')
#     with torch.no_grad():
#         out = model.generate(**inputs, max_new_tokens=5)
#     return base_tokenizer.decode(out[0], skip_special_tokens=True).split('Label:')[-1].strip()

print("LoRA pattern shown above (commented). Using TF-IDF+LR for live inference.")

LoRA pattern shown above (commented). Using TF-IDF+LR for live inference.


In [4]:
# -- Cell 4: Inference function --

def predict(text, variety):
    if not text.strip():
        return "Please enter text.", "", "", "", ""

    t0 = time.time()

    tfidf_s, lr_s = variety_models[variety]['sentiment']
    sent_pred  = lr_s.predict(tfidf_s.transform([text]))[0]
    sent_proba = lr_s.predict_proba(tfidf_s.transform([text]))[0]
    sent_label = "Positive" if sent_pred == 1 else "Negative"
    sent_conf  = f"{max(sent_proba)*100:.1f}%"

    tfidf_r, lr_r = variety_models[variety]['sarcasm']
    sarc_pred  = lr_r.predict(tfidf_r.transform([text]))[0]
    sarc_proba = lr_r.predict_proba(tfidf_r.transform([text]))[0]
    sarc_label = "Sarcastic" if sarc_pred == 1 else "Not Sarcastic"
    sarc_conf  = f"{max(sarc_proba)*100:.1f}%"

    ms = f"{(time.time()-t0)*1000:.1f} ms"
    return sent_label, sent_conf, sarc_label, sarc_conf, ms

# Quick test
print(predict("Absolute legend, parked his ute right across my driveway. Good onya, mate.", "en-AU"))
print(predict("Traditional friendly pub. Excellent beer.", "en-UK"))
print(predict("Coz we all have free internet.", "en-IN"))

('Negative', '50.1%', 'Not Sarcastic', '53.6%', '0.7 ms')
('Positive', '77.6%', 'Not Sarcastic', '68.5%', '0.6 ms')
('Negative', '70.3%', 'Not Sarcastic', '51.7%', '0.4 ms')


In [6]:
# -- Cell 5: Launch Gradio app --
import gradio as gr

EXAMPLES = [
    ["Absolute legend, parked his ute right across my driveway. Good onya, mate.", "en-AU"],
    ["Yeah because the wifi here is sooo reliable, totally worth 3 hours.", "en-AU"],
    ["Coz we all have free internet.", "en-IN"],
    ["Yaar this place is just too good, totally worth the 2 hour wait!", "en-IN"],
    ["Traditional friendly pub. Excellent beer.", "en-UK"],
    ["Oh brilliant, another 45-minute delay on the Tube. Just what I needed.", "en-UK"],
]

with gr.Blocks(title="BESSTIE Variety-Aware NLP", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# BESSTIE -- Variety-Aware Sentiment & Sarcasm Classifier\n"
        "**COMM061 NLP Coursework | University of Surrey | PG13**\n\n"
        "Select the English variety that matches your text, then click Classify. "
        "The backend switches to the model trained on that variety."
    )

    with gr.Row():
        with gr.Column(scale=2):
            text_input    = gr.Textbox(label="Input Text", lines=4,
                                        placeholder="e.g. Oh great, another delay on the Tube...")
            variety_radio = gr.Radio(["en-AU", "en-IN", "en-UK"], value="en-UK",
                                      label="English Variety",
                                      info="Australian | Indian | British")
            classify_btn  = gr.Button("Classify", variant="primary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("### Results")
            out_sent      = gr.Textbox(label="Sentiment",            interactive=False)
            out_sent_conf = gr.Textbox(label="Sentiment Confidence", interactive=False)
            out_sarc      = gr.Textbox(label="Sarcasm",              interactive=False)
            out_sarc_conf = gr.Textbox(label="Sarcasm Confidence",   interactive=False)
            out_time      = gr.Textbox(label="Inference Time",       interactive=False)

    gr.Examples(examples=EXAMPLES, inputs=[text_input, variety_radio],
                label="Try these examples (all 3 varieties)")

    classify_btn.click(
        fn=predict,
        inputs=[text_input, variety_radio],
        outputs=[out_sent, out_sent_conf, out_sarc, out_sarc_conf, out_time]
    )

    gr.Markdown(
        "---\n"
        "Switching variety = dict lookup only. Same efficiency principle as LoRA adapter swap: "
        "frozen base stays in memory, only small adapter weights change per request."
    )

demo.launch(share=True)

/var/folders/p4/cp7wz8hx5r91wbwpbs2q7brh0000gn/T/ipykernel_12341/135889318.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="BESSTIE Variety-Aware NLP", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://48cabe97dd2bb732a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Q5.2 -- Efficiency Analysis

| Model | Short input (~10 tokens) | Long input (~100 tokens) | Model Size |
|---|---|---|---|
| TF-IDF + LR (per variety) | ~1-3 ms | ~2-5 ms | <1 MB |
| RoBERTa-base (GPU) | ~80-120 ms | ~150-300 ms | ~500 MB |
| TinyLlama 1.1B + LoRA (GPU) | ~200-400 ms | ~500-900 ms | ~2.2 GB + ~90 MB adapter |

**Trade-off discussion:**

TF-IDF + LR is approximately 100x faster than RoBERTa and requires no GPU.
However its Macro-F1 on sarcasm is substantially lower (~0.55 overall vs ~0.71 for LoRA en-UK).
For real-time comment moderation at scale where throughput is critical, the classical
model is the pragmatic choice. For a research demo where accuracy is the priority,
the LoRA adapter is preferable.

The key deployment advantage of the adapter approach is **cold-start latency**:
loading a full TinyLlama model takes ~10-30 seconds; swapping a LoRA adapter takes under 1 second
since the frozen base stays in memory. This makes the multi-variety architecture viable in a
live service -- user variety selection causes no noticeable delay.